# 🤖 Notebook 06: Drought Risk Modeling

**Objective:** Build ML models to classify drought risk and predict water stress.

**Models:**
- Random Forest Classifier (drought risk categories)
- XGBoost Classifier (drought risk categories)
- XGBoost Regressor (water stress score prediction)
- Ridge Regression (baseline)

**Techniques:**
- Feature engineering (lag features, rolling statistics)
- Temporal train/test split
- Hyperparameter tuning (GridSearchCV)
- SHAP explainability analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import logging
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

import config
from src.models.drought_classifier import DroughtClassifier
from src.models.stress_predictor import StressPredictor

plt.style.use('dark_background')
print('✅ Setup complete')

In [ ]:
# Load consolidated data
df = pd.read_csv(config.CONSOLIDATED_CSV)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year

print(f'Dataset: {df.shape}')
print(f'Target distribution:')
print(df['drought_risk_class'].value_counts())

## Part 1: Drought Risk Classification

In [ ]:
# Initialize classifier
classifier = DroughtClassifier()

# Prepare data (temporal split)
X_train, X_test, y_train, y_test, features = classifier.prepare_data(df)

print(f'\nFeatures ({len(features)}): {features}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train all models with hyperparameter tuning
models = classifier.train_all(X_train, y_train, tune=True)
print(f'\n✅ Trained {len(models)} models: {list(models.keys())}')

In [ ]:
# Evaluate
results = classifier.evaluate(X_test, y_test)

# Results comparison table
comparison = pd.DataFrame({
    name: {'F1': r['f1_weighted'], 'Accuracy': r['accuracy']}
    for name, r in results.items()
}).T
print('\nModel Comparison:')
print(comparison.to_string())

In [ ]:
# Confusion Matrix
best_name = classifier.best_model_name
best_model = classifier.models[best_name]
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=classifier.label_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importance = classifier.get_feature_importance()
if importance is not None:
    fig = px.bar(
        importance.head(15), x='importance', y='feature',
        orientation='h', title=f'Top 15 Feature Importances — {best_name}',
        color='importance', color_continuous_scale='Viridis',
    )
    fig.update_layout(
        template='plotly_dark', height=500,
        yaxis=dict(autorange='reversed'),
    )
    fig.show()

In [ ]:
# SHAP Analysis
try:
    import shap
    shap_values, X_sample = classifier.get_shap_values(X_test, max_samples=300)
    
    if shap_values is not None:
        plt.figure(figsize=(12, 8))
        if isinstance(shap_values, list):
            # Multi-class: show summary for all classes
            shap.summary_plot(shap_values, X_sample, plot_type='bar',
                              class_names=classifier.label_encoder.classes_, show=False)
        else:
            shap.summary_plot(shap_values, X_sample, show=False)
        plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
except ImportError:
    print('SHAP not installed. Install with: pip install shap')

In [ ]:
# Save best classifier
model_path = classifier.save_model()
print(f'\n✅ Best model ({best_name}) saved to: {model_path}')

## Part 2: Water Stress Prediction (Regression)

In [ ]:
# Initialize predictor
predictor = StressPredictor()

# Create lag features
df_lags = predictor.create_lag_features(df)
print(f'Dataset with lags: {df_lags.shape}')

# Prepare data
X_train_r, X_test_r, y_train_r, y_test_r, features_r = predictor.prepare_data(df_lags)
print(f'Train: {X_train_r.shape}, Test: {X_test_r.shape}')

In [ ]:
# Train regression models
reg_models = predictor.train_all(X_train_r, y_train_r, tune=True)
print(f'\n✅ Trained {len(reg_models)} regression models')

In [ ]:
# Evaluate
reg_results = predictor.evaluate(X_test_r, y_test_r)

# Results table
reg_comparison = pd.DataFrame(reg_results).T
print('\nRegression Model Comparison:')
print(reg_comparison.to_string())

In [ ]:
# Actual vs Predicted scatter
best_reg = predictor.best_model_name
y_pred_r = predictor.predict(X_test_r)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test_r, y=y_pred_r, mode='markers',
    marker=dict(size=4, opacity=0.5, color='#667eea'),
    name='Predictions'
))
# Perfect prediction line
min_val = min(y_test_r.min(), y_pred_r.min())
max_val = max(y_test_r.max(), y_pred_r.max())
fig.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Perfect Prediction'
))

fig.update_layout(
    title=f'Actual vs Predicted — {best_reg} (R²={reg_results[best_reg]["R2"]:.3f})',
    xaxis_title='Actual', yaxis_title='Predicted',
    template='plotly_dark', height=500,
)
fig.show()

In [ ]:
# Save predictor
pred_path = predictor.save_model()
print(f'✅ Predictor saved to: {pred_path}')

In [ ]:
# Summary
print('\n' + '='*60)
print('MODEL TRAINING SUMMARY')
print('='*60)
print(f'\nClassification (best: {classifier.best_model_name}):')
for name, r in results.items():
    print(f'  {name}: F1={r["f1_weighted"]:.4f}, Acc={r["accuracy"]:.4f}')
print(f'\nRegression (best: {predictor.best_model_name}):')
for name, r in reg_results.items():
    print(f'  {name}: RMSE={r["RMSE"]:.4f}, R²={r["R2"]:.4f}')